# 03 - Backbone pilot (decision gate G1)
Compares the fixed-prior per-step objective with the progressive loss (Bansal et al. 2022) for
the GLWM-2 backbone on addition (T1) and segmented parity (T2). The runs themselves are launched
from a terminal with `scripts/run_sweep.py configs/pilot_addition.yaml`; this notebook only reads
their results. The objective for the main grid is chosen per task on OOD-val and ID-val only.

**G1 (pre-registered):** stable-oracle accuracy >= 90% on OOD-both instances with d* >= 40
(4x the training maximum), on both tasks, averaged over seeds.

In [ ]:
# parameters
CONFIG = "configs/pilot_addition.yaml"
G1_DEPTH, G1_ACC = 40, 0.90

In [ ]:
%matplotlib inline
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from glwm2 import PROJECT_ROOT, RESULTS_DIR, PAPER_DIR
from glwm2.eval.plotting import set_style
set_style()
WRITTEN = []  # files this notebook writes

In [ ]:
from glwm2.eval.results import configs_from_yaml, finished, metric_rows, instance_frame, step_acc
from glwm2.utils.io import write_json
cfgs = configs_from_yaml(PROJECT_ROOT / CONFIG)
done = finished(cfgs)
print(f"{len(done)}/{len(cfgs)} pilot runs finished")
df = metric_rows(done)
ref = df[df.rule.isin(["oracle_stable", "oracle_iter", "oracle_lb", "fixed_T"])]
tab = ref.pivot_table(index=["task", "objective", "rule"], columns="split", values="acc", aggfunc="mean")
tab.round(3)

In [ ]:
# G1: stable-oracle accuracy on deep OOD-both instances, per run
g1_rows = []
for c in done:
    inst = instance_frame(c, "ood_both", ["oracle_stable"])
    deep = inst[inst.depth >= G1_DEPTH]
    g1_rows.append(dict(task=c.task, objective=c.objective, seed=c.seed, n_deep=len(deep), acc_deep=deep.correct.mean()))
g1 = pd.DataFrame(g1_rows)
g1_summary = g1.groupby(["task", "objective"]).acc_deep.agg(["mean", "std", "count"])
g1_summary

In [ ]:
# objective selection on validation splits only (OOD-val, then ID-val as tie-break)
sel = (df[df.rule == "oracle_stable"].pivot_table(index=["task", "objective"], columns="split", values="acc", aggfunc="mean")
       [["ood_val", "id_val"]])
choice = {}
for task in sel.index.get_level_values(0).unique():
    s = sel.loc[task].sort_values(["ood_val", "id_val"], ascending=False)
    choice[task] = s.index[0]
print("selected objective:", choice)
g1_pass = {task: bool(g1_summary.loc[(task, obj), "mean"] >= G1_ACC) for task, obj in choice.items()}
print("G1 passed per task:", g1_pass, "-> overall", all(g1_pass.values()))

In [ ]:
from glwm2.eval.plotting import savefig, SPLIT_COLORS
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.5))
for ax, task in zip(axes, ["addition", "parity"]):
    for obj, ls in [("fixed_prior", "-"), ("progressive", "--")]:
        runs = [c for c in done if c.task == task and c.objective == obj]
        if not runs:
            continue
        for split in ["id_test", "ood_depth", "ood_length", "ood_both"]:
            curves = [step_acc(c)[split] for c in runs]
            T = min(len(v) for v in curves)
            ax.plot(np.arange(1, T + 1), np.mean([v[:T] for v in curves], 0), ls, color=SPLIT_COLORS[split],
                    label=f"{split} ({obj})" if task == "addition" else None)
    ax.set_xscale("log"); ax.set_xlabel("step t"); ax.set_ylabel("exact-match accuracy at step t"); ax.set_title(task)
axes[0].legend(fontsize=6, frameon=False, ncol=1)
WRITTEN.append(savefig(fig, "figA_step_accuracy"))

In [ ]:
# Fig. 3: stable-oracle accuracy against depth and against length (selected objective, all splits pooled)
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.5))
for task, color in [("addition", "#D55E00"), ("parity", "#0072B2")]:
    runs = [c for c in done if c.task == task and c.objective == choice.get(task)]
    frames = [instance_frame(c, s, ["oracle_stable"]) for c in runs for s in ["id_test", "ood_depth", "ood_length", "ood_both"]]
    if not frames:
        continue
    inst = pd.concat(frames)
    for ax, col, bins in [(axes[0], "depth", [0, 2, 4, 8, 12, 16, 20, 32, 48, 64, 96, 128]), (axes[1], "length", [4, 8, 12, 16, 20, 64, 96, 128, 192, 256])]:
        inst["bin"] = pd.cut(inst[col], bins, include_lowest=True)
        g = inst.groupby("bin", observed=True).correct.mean()
        ax.plot([b.right for b in g.index], g.values, "o-", color=color, label=task)
        ax.set_xscale("log", base=2); ax.set_xlabel(col if col == "length" else "depth $d^*$"); ax.set_ylim(0, 1.02)
axes[0].axvline(10, color="grey", lw=0.8, ls=":"); axes[1].axvline(20, color="grey", lw=0.8, ls=":")
axes[0].set_ylabel("stable-oracle accuracy"); axes[0].legend(frameon=False)
WRITTEN.append(savefig(fig, "fig3_oracle_vs_depth_length"))

In [ ]:
# position-by-step correctness for one deep OOD-both addition instance
from glwm2.train.trainer import load_model
from glwm2.data.splits import load_split
from glwm2.eval.trajectory import per_position_rho
c = next(c for c in done if c.task == "addition" and c.objective == choice["addition"])
model = load_model(c)
data = load_split("addition", "ood_both")
k = int(torch.argsort(data["depth"])[len(data["depth"]) // 2])
x, y = data["x"][k:k+1].cuda(), data["y"][k:k+1].cuda()
L = int(data["length"][k]); rho = per_position_rho("addition", {k_: v[k:k+1] for k_, v in data.items()})[0, :L+1].numpy()
with torch.no_grad():
    T = int(data["depth"][k]) + 16
    pc = torch.stack([(o["logits"].argmax(-1) == y)[0, :L+1] for o in model.rollout(x, T)]).cpu().numpy()
fig, ax = plt.subplots(figsize=(7.0, 2.4))
ax.imshow(pc.T, aspect="auto", origin="lower", cmap="Greys_r", interpolation="nearest", extent=[0.5, T + 0.5, -0.5, L + 0.5])
ax.plot(rho, np.arange(L + 1), color="#D55E00", lw=1, label=r"lower bound $\rho_i$")
ax.set_xlabel("step t"); ax.set_ylabel("output position i"); ax.legend(frameon=False, loc="upper left")
ax.set_title(f"addition, L={L}, d*={int(data['depth'][k])}: white = correct")
WRITTEN.append(savefig(fig, "figA_position_by_step"))

In [ ]:
WRITTEN.append(write_json(RESULTS_DIR / "pilot_summary.json", {
    "objective": choice, "g1": g1_summary.reset_index().to_dict(orient="records"), "g1_pass": g1_pass,
    "table": tab.reset_index().to_dict(orient="records")}))

In [ ]:
print("Files written by this notebook:")
for p in WRITTEN:
    print(" ", Path(p).relative_to(PROJECT_ROOT))